# 11 - Financial Safety (Lease Lifecycle)

ducto charges **after** an AI operation completes — but that is exactly when things can go wrong. If you only check the balance *before* the call and debit *after* it returns, you have a race: between the check and the debit, other concurrent operations can also pass the check, and a single expensive (or runaway agentic) call can finish costing far more than you estimated. The result is **unrecoverable debt** — a balance that has already gone negative on work that is already done.

The fix is an atomic **lease** taken *before* the work. A lease holds credits against `available = balance − Σ(active holds)`, so concurrent operations actually see each other. Every operation follows the same shape: **`reserve`** (place a hold) → **do the work** → **`settle`** (charge the actual cost) or **`release`** (cancel, charging nothing). Admission is the *only* place limits are enforced; `settle` is **de-clamped** — it always bills the real cost, even if that exceeds the hold.

Two presets cover the common cases:

- **`strict_prepaid`** (the default): floor >= 0, structurally zero debt. You size the hold at the **worst case**, so admission can never let the balance go negative.
- **`overdraft`**: a negative floor for trusted/paid users with auto-reload. Admission is still bounded by the floor, but `settle` bills the full actual cost even past it; you reconcile with `add_credits`.

This notebook walks the whole model on a real PostgreSQL store. All money is `Decimal`.

In [ ]:
import uuid
from decimal import Decimal

from ducto import (
    CreditManager,
    UsageMetrics,
    ConcurrencyLimitError,
    FeatureNotEntitledError,
    InsufficientCreditsError,
)
from ducto.events import CreditEvent, CreditEventEmitter
from ducto.interface.models import PricingConfigData, PlanDefinition, OperationPolicy
from shared import start_postgres_store, cleanup

# A temporary Postgres cluster with the ducto schema already applied.
# PostgresStore uses UUID user ids, so every demo user below is a fresh uuid4().
store, pgdata = start_postgres_store()
print("\u2714 PostgresStore ready.")

## 1. `strict_prepaid`: reserve → settle, with worst-case sizing

We build a `CreditManager` with the default `strict_prepaid` policy and a small per-token price. The key discipline in strict mode: **size the hold at the worst case** you might bill. Admission subtracts that hold from `available`, so the balance can never be driven negative by work in flight. At `settle` time you bill the *actual* cost — which is usually less than the worst case, so the difference is automatically freed.

Below, a `gpt-4` call is reserved at its worst-case token budget, then settled at the (smaller) actual usage.

In [ ]:
# strict_prepaid is the default policy; min_balance=0 keeps the arithmetic obvious.
manager = CreditManager(store=store, policy="strict_prepaid")
manager.publish_pricing_from_dict({
    "version": 1,
    "models": {"gpt-4": "input_tokens * 0.01 + output_tokens * 0.03"},
    "min_balance": 0,
})

user = str(uuid.uuid4())
manager.add_credits(user, Decimal("100"), "signup_bonus")

# Size the hold at the WORST CASE we might bill for this call.
worst_case = UsageMetrics(model="gpt-4", input_tokens=1000, output_tokens=1000)  # cost 40
lease = manager.reserve(user, worst_case, operation_type="chat")
print(f"  Lease:     {lease.lease_id}")
print(f"  Held:      {lease.amount}")          # 40 reserved against available
print(f"  Available: {lease.available}")        # 100 - 40 = 60 left for other work
print(f"  Mode:      {lease.billing_mode}")

# ... do the AI work ... it turned out cheaper than the worst case.
actual = UsageMetrics(model="gpt-4", input_tokens=500, output_tokens=200)  # cost 11
ded = manager.settle(user, lease.lease_id, actual)
print(f"  Charged:   {ded.amount}")             # 11, the ACTUAL cost (not the 40 hold)
print(f"  Balance:   {ded.balance_after}")        # 100 - 11 = 89; the unused 29 was freed
assert ded.balance_after == Decimal("89.00")

### Releasing a lease (work aborted)

If the work fails or is cancelled, call `release` instead of `settle`. Nothing is charged and the hold is returned to `available`. `release` is **idempotent**: calling it twice (or after a settle) is safe and reports a `reason` rather than raising.

In [ ]:
lease = manager.reserve(user, Decimal("20"), operation_type="chat")
print(f"  Reserved 20, available now: {manager.get_available(user).available}")  # 89 - 20 = 69

# The operation failed — cancel the hold, charge nothing.
rel = manager.release(user, lease.lease_id)
print(f"  Released:  {rel.released}  reason={rel.reason}")
print(f"  Available restored: {manager.get_available(user).available}")           # back to 89

# Idempotent: a second release does not raise, it just reports the state.
rel2 = manager.release(user, lease.lease_id)
print(f"  Second release: released={rel2.released} reason={rel2.reason}")
assert manager.get_balance(user).balance == Decimal("89.00")

## 2. `run_billed`: the one-call shortcut

Wiring reserve → work → settle/release by hand is repetitive and easy to get wrong (e.g. forgetting to release on an exception). `run_billed` does it for you: pass an `estimate` (the worst-case hold) and a zero-arg `do_work` callable that returns `(result, actual)`. On success it settles the actual cost; **on any exception it auto-releases the lease** and re-raises.

In [ ]:
def do_work():
    # Run the real operation, then report what it actually cost.
    answer = "the AI response"
    actual = UsageMetrics(model="gpt-4", input_tokens=300, output_tokens=100)  # cost 6
    return answer, actual

out = manager.run_billed(
    user,
    estimate=UsageMetrics(model="gpt-4", input_tokens=1000, output_tokens=1000),  # worst-case hold 40
    do_work=do_work,
    operation_type="chat",
)
print(f"  Result:   {out['result']!r}")
print(f"  Charged:  {out['deduction'].amount}")          # 6, the actual cost
print(f"  Balance:  {out['deduction'].balance_after}")     # 89 - 6 = 83
assert out["deduction"].balance_after == Decimal("83.00")

## 3. `max_concurrent`: blocking a double-submit

A classic failure is the impatient user clicking *Send* twice, or a client retrying before the first request finished. Because each in-flight operation holds a lease, ducto can cap the number of **simultaneously-active leases per operation type** with `max_concurrent`. The second admission for the same operation type is rejected with `ConcurrencyLimitError` — no balance leaks, no double charge.

In [ ]:
# Only one 'chat' operation may be in flight at a time for this manager.
conc_mgr = CreditManager(store=store, policy="strict_prepaid", max_concurrent=1)
conc_mgr.publish_pricing_from_dict({
    "version": 1,
    "models": {"_default": "input_tokens * 1"},
    "min_balance": 0,
})

u = str(uuid.uuid4())
conc_mgr.add_credits(u, Decimal("100"))

first = conc_mgr.reserve(u, Decimal("10"), operation_type="chat")
print(f"  First lease admitted: {first.lease_id}")

# The double-submit: a second concurrent 'chat' is refused, not silently charged.
try:
    conc_mgr.reserve(u, Decimal("10"), operation_type="chat")
    print("  (unexpected) second admission succeeded")
except ConcurrencyLimitError as e:
    print(f"  Second submit blocked: ConcurrencyLimitError")

# Releasing the first frees the slot again.
conc_mgr.release(u, first.lease_id)
second = conc_mgr.reserve(u, Decimal("10"), operation_type="chat")
print(f"  After release, slot free: {second.lease_id}")
conc_mgr.release(u, second.lease_id)

## 4. The `overdraft` preset

Strict mode guarantees zero debt, but sometimes you *want* to let trusted, paid users run past their balance and reconcile later (typically via auto-reload). The `overdraft` preset admits down to a **negative `overdraft_floor`** and, because `settle` is de-clamped, bills the **full actual cost even when it pushes the balance below the floor**. The work is already done — you never silently under-bill.

After the balance goes negative, a **new admission is rejected** (available has dropped past the floor) until you reconcile with `add_credits`.

In [ ]:
# Paid user: overdraft allowed down to -50.
od_mgr = CreditManager(store=store, policy="overdraft", overdraft_floor=Decimal("-50"))
od_mgr.publish_pricing_from_dict({
    "version": 1,
    "models": {"_default": "input_tokens * 1"},
    "min_balance": 0,
})

pu = str(uuid.uuid4())
od_mgr.add_credits(pu, Decimal("0"))  # start at zero balance

lease = od_mgr.reserve(pu, Decimal("10"))  # small estimate, within the floor
print(f"  Reserved estimate: {lease.amount}, mode={lease.billing_mode}")

# The work actually cost 60 — de-clamped settle bills it in full, past the -50 floor.
ded = od_mgr.settle(pu, lease.lease_id, Decimal("60"))
print(f"  Charged actual:    {ded.amount}")
print(f"  Balance (negative):{ded.balance_after}")   # -60
assert ded.balance_after == Decimal("-60")

# A NEW operation is now refused — available has dropped below the floor.
try:
    od_mgr.reserve(pu, Decimal("1"))
    print("  (unexpected) admission succeeded while overdrawn")
except InsufficientCreditsError:
    print("  New admission refused while overdrawn (bounded by the floor)")

# Auto-reload / top-up reconciles the negative balance (a 'purchase' top-up).
res = od_mgr.add_credits(pu, Decimal("200"), "purchase")
print(f"  After reload:      {res.new_balance}")     # -60 + 200 = 140
assert res.new_balance == Decimal("140")

## 5. Feature gating

Some operations should only be available on certain plans (e.g. autonomous *agentic* runs are Pro-only). Define `features` on the plan, then pass `required_feature` to `reserve`. A user whose plan lacks the feature is rejected at admission with `FeatureNotEntitledError` — before any work or hold happens.

Plans live in the active pricing config; assign one to a user with `set_user_plan`.

In [ ]:
feat_mgr = CreditManager(store=store, policy="strict_prepaid")
feat_mgr.publish_pricing(PricingConfigData(
    models={"_default": "input_tokens * 1"},
    min_balance=Decimal(0),
    plans={
        "free": PlanDefinition(id="free", name="Free", features={"chat": True}),
        "pro": PlanDefinition(id="pro", name="Pro", features={"chat": True, "agentic": True}),
    },
))

free_user = str(uuid.uuid4())
pro_user = str(uuid.uuid4())
for uid in (free_user, pro_user):
    feat_mgr.add_credits(uid, Decimal("100"))
feat_mgr.set_user_plan(free_user, "free")
feat_mgr.set_user_plan(pro_user, "pro")

# Free user cannot run an 'agentic' operation — blocked at admission.
try:
    feat_mgr.reserve(free_user, Decimal("10"), required_feature="agentic")
    print("  (unexpected) free user admitted to agentic")
except FeatureNotEntitledError:
    print("  Free user blocked from 'agentic' (FeatureNotEntitledError)")

# Pro user is entitled.
pro_lease = feat_mgr.reserve(pro_user, Decimal("10"), required_feature="agentic")
print(f"  Pro user admitted to agentic: {pro_lease.lease_id}")
feat_mgr.release(pro_user, pro_lease.lease_id)

## 6. Multi-level low-balance alerts + reload hook

Pass `low_balance_thresholds` (sorted internally high → low) to get **edge-triggered** alerts: each level fires once on the descent that crosses it, and re-arms only after a top-up climbs back above it. The `on_low_balance` hook is an async-safe, **non-blocking** callable — perfect for a payment-provider-agnostic reload or notification. A handler that raises never breaks the operation.

We use the `overdraft` preset here only so the balance can descend freely through every threshold.

In [ ]:
fired = []

def on_low_balance(event: CreditEvent):
    # Payment-provider-agnostic: enqueue a reload, send an email, page on-call, etc.
    level = event.data["threshold"]
    balance = event.data["balance"]
    fired.append(level)
    print(f"  [hook] balance {balance} crossed threshold {level} — triggering reload")

emitter = CreditEventEmitter()
lb_mgr = CreditManager(
    store=store,
    emitter=emitter,
    policy="overdraft",
    overdraft_floor=Decimal("0"),
    low_balance_thresholds=[Decimal("50"), Decimal("20"), Decimal("10")],
    on_low_balance=on_low_balance,
)
lb_mgr.publish_pricing_from_dict({
    "version": 1,
    "models": {"_default": "input_tokens * 1"},
    "min_balance": 0,
})

lbu = str(uuid.uuid4())
lb_mgr.add_credits(lbu, Decimal("100"))

def charge(amount):
    lease = lb_mgr.reserve(lbu, Decimal(amount))
    lb_mgr.settle(lbu, lease.lease_id, Decimal(amount))

charge(55)  # 100 -> 45 : crosses 50
charge(30)  # 45  -> 15 : crosses 20
charge(7)   # 15  -> 8  : crosses 10
print(f"  Thresholds fired (each once): {fired}")
assert fired == [Decimal("50"), Decimal("20"), Decimal("10")]

## 7. Advisory reads: `get_available` and `can_afford`

For **UI only** — disabling a button, showing a balance — use the non-locking advisory reads. They are fast and may be slightly stale; they are **never** an admission gate. Only `reserve` is authoritative.

- `get_available` returns `balance`, `reserved` (sum of active holds), and `available`.
- `can_afford` reports whether a worst-case amount would fit (and *why* not), including feature gates.

In [ ]:
adv_user = str(uuid.uuid4())
manager.add_credits(adv_user, Decimal("100"))
hold = manager.reserve(adv_user, Decimal("30"), operation_type="chat")

avail = manager.get_available(adv_user)
print(f"  balance={avail.balance} reserved={avail.reserved} available={avail.available}")
assert avail.available == Decimal("70")  # 100 - 30 held

# Would a 50-credit worst-case op fit right now? (70 available, yes.)
ok = manager.can_afford(adv_user, Decimal("50"))
print(f"  can_afford(50): affordable={ok.affordable} available={ok.available} worst_case={ok.worst_case}")

# A 90-credit worst-case op would NOT fit — advisory 'no' with a reason.
no = manager.can_afford(adv_user, Decimal("90"))
print(f"  can_afford(90): affordable={no.affordable} reason={no.reason}")
assert no.affordable is False and no.reason == "insufficient_credits"

manager.release(adv_user, hold.lease_id)

## Recap

- **`reserve` → `settle`/`release`** is the safe shape for charge-after billing; the lease holds against `available = balance − Σ(active holds)`, so concurrency is real.
- **`strict_prepaid`** sizes holds at the worst case and gives **structural zero debt**; **`overdraft`** allows bounded negative admission and full de-clamped billing for paid users who auto-reload.
- **`max_concurrent`** stops double-submits; **`required_feature`** gates operations by plan.
- **`run_billed`** wires the whole flow (and auto-releases on failure) in one call.
- **`low_balance_thresholds` + `on_low_balance`** give edge-triggered, non-blocking, provider-agnostic reload hooks.
- **`get_available` / `can_afford`** are advisory UI reads, never admission gates.

See the [Financial Safety](/docs/financial-safety) doc page for the conceptual reference and the TypeScript equivalents.

In [ ]:
cleanup(pgdata)